# **AfriLink SDK — End-to-End Finetune Demo**

This notebook walks through the **complete workflow** of the AfriLink SDK:

| Step | What | API |
|------|------|-----|
| 1 | Install SDK + headless browser | `pip install afrilink-sdk` |
| 2 | Authenticate (DataSpires + CINECA) | `client.authenticate()` |
| 3 | Browse available models & datasets | `client.list_available_models()` |
| 4 | Prepare dataset | pandas DataFrame |
| 5 | Submit finetune job | `client.finetune(...).run(wait=True)` |
| 6 | Download trained weights | `client.download_model(job_id, dir)` |
| 7 | Load adapter & run inference | `PeftModel.from_pretrained(base, dir)` |

**Prerequisites:** A free DataSpires account — sign up at https://dataspires.com

You can explore how it works with the example input or rerun the cells with your own input.

---
## 1. **Install the AfriLink SDK**

AfriLink SDK gives you one-line access GPUs, model and datasets, all ready to use directly from a Google Colab notebook. Authenticate, submit LoRA finetune jobs, download trained weights, and run inference — all without leaving your notebook.

Just run `pip install afrilink-sdk` and follow the rest of the steps below!

In [ ]:
!pip install -q afrilink-sdk

import afrilink
print(f"AfriLink SDK v{afrilink.__version__} ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.4 MB/s eta 0:00:00
CINECA credentials provisioned successfully
AfriLink SDK v0.1.3 ready


---
## 2. **Authenticate with your DataSpires Account**

A single `client.authenticate()` call handles everything:

1. **DataSpires** — you'll be prompted for your email & password (for logging usage data)

After this cell you have an SSH-authenticated session to High Performance Compute with tried and tested links to models and data.

In [ ]:
from afrilink import AfriLinkClient

client = AfriLinkClient()
client.authenticate()  # Interactive: prompts for DataSpires credentials


DataSpires Authentication Required
Your DataSpires account is used for usage tracking and billing.
Don't have an account? Sign up at https://dataspires.com

DataSpires Email: ianomunga@gmail.com
DataSpires Password: ··········

Authenticating with DataSpires...
  Authenticated as: ianomunga@gmail.com
  User ID: ccab4bb3-2705-4979-be14-2ca8bf871fbb
  Balance: $8.57

------------------------------------------------------------

Authenticating with CINECA...
(Using org-wide shared credentials)
Starting headless authentication...
Installing Smallstep CLI...
Extracting...
Installed to /root/.local/bin/step
Bootstrapping Smallstep CA...
CA bootstrapped successfully
Installing Selenium...
Installing Google Chrome...
[1/4] Pre-launching headless browser...
      Browser ready.
[2/4] Starting step ssh login...
      [step] ✔ Provisioner: cineca-hpc (OIDC) [client: step-ca]
      [step] Your default web browser has been opened to visit:
      [step] https://sso.hpc.cineca.it/realms/CINECA-HPC/p

CinecaAuthResult(success=True, cert_path='ssh-agent', key_path='ssh-agent', expires_at=None, username='iomunga0', error=None)

In [ ]:
# Quick sanity check for checking that the high performance compute is linked well :)
code, out, err = client.run_command("hostname && echo $WORK")
print(out)

login05.leonardo.local
/leonardo_work/AIH4A_dataspire



---
## 3. **Browse AfriLink's Available Models & Datasets**

The SDK ships with a registry of pre-configured models (text LLMs and vision-language models) and datasets.

For this demo, the model that's ready for full testing is the `qwen2.5-0.5b` Multimodal model.

In [ ]:
from afrilink import list_models, list_datasets

models = list_models()

print("TEXT MODELS")
print("-" * 65)
for m in models:
    if m["type"] == "text":
        print(f"  {m['id']:20} {m['parameters_b']:.2f}B  {m['min_gpu_memory_gb']:>2}GB VRAM  {m['name']}")

print("\nVISION-LANGUAGE MODELS")
print("-" * 65)
for m in models:
    if m["type"] == "vision":
        print(f"  {m['id']:20} {m['parameters_b']:.2f}B  {m['min_gpu_memory_gb']:>2}GB VRAM  {m['name']}")

print("\nDATASETS")
print("-" * 65)
for d in list_datasets():
    n = d.get('num_examples') or 'N/A'
    print(f"  {d['id']:25} {str(n):>8} examples  {d['name']}")

TEXT MODELS
-----------------------------------------------------------------
  qwen2.5-0.5b         0.50B   4GB VRAM  Qwen 2.5 0.5B
  gemma-3-270m         0.27B   2GB VRAM  Gemma 3 270M
  ministral-3b         3.30B   8GB VRAM  Ministral 3B Reasoning
  llama-3.2-1b         1.00B   4GB VRAM  Llama 3.2 1B
  deepseek-r1-1.5b     1.50B   6GB VRAM  DeepSeek R1 Distill Qwen 1.5B

VISION-LANGUAGE MODELS
-----------------------------------------------------------------
  llava-1.5-7b         7.00B  16GB VRAM  LLaVA 1.5 7B
  moondream2           1.90B   8GB VRAM  Moondream 2
  florence-2-base      0.23B   4GB VRAM  Florence 2 Base
  smolvlm-256m         0.26B   2GB VRAM  SmolVLM 256M Instruct
  internvl2-1b         1.00B   4GB VRAM  InternVL2 1B

DATASETS
-----------------------------------------------------------------
  multilingual-thinking          N/A examples  Multilingual Thinking
  alpaca                       52000 examples  Stanford Alpaca
  dolly                        15000 examples

In [ ]:
# Check resource requirements for a specific model
client.get_model_requirements("qwen2.5-0.5b", "low")

{'model': 'qwen2.5-0.5b',
 'model_type': 'text',
 'training_mode': 'low',
 'recommended_gpus': 1,
 'min_memory_gb': 4,
 'parameters_b': 0.5,
 'context_length': 32768,
 'requires_vision_encoder': False}

---
## 4. **Prepare a Small Example Dataset**

The SDK accepts all these kinds of dataset configurations:
- **pandas DataFrame** with a `text` column (Alpaca-style prompt+response in one string)
- **HuggingFace Dataset**
- **File path** to a local JSONL/CSV

Below we create a small example. In production you'd use a real dataset loaded from your notebook with `pandas`

In [ ]:
import pandas as pd

# Alpaca-style: each row is a self-contained prompt + response
rows = [
    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\nWhat is machine learning?\n\n"
    "### Response:\nMachine learning is a branch of artificial intelligence that "
    "enables systems to learn patterns from data and improve with experience.",

    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\nExplain gradient descent in one sentence.\n\n"
    "### Response:\nGradient descent is an optimisation algorithm that iteratively "
    "adjusts model parameters in the direction that minimises the loss function.",

    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\nWhat is a neural network?\n\n"
    "### Response:\nA neural network is a computing architecture composed of "
    "interconnected layers of nodes that learns to map inputs to outputs.",
]

dataset = pd.DataFrame({"text": rows})
print(f"Dataset: {len(dataset)} rows")
print(dataset.head())

Dataset: 3 rows
                                                text
0  Below is an instruction that describes a task....
1  Below is an instruction that describes a task....
2  Below is an instruction that describes a task....


---
## 5. **Submit Finetune Job to AfriLink's HPC**

`client.finetune()` creates the job; `.run(wait=True)` submits it to SLURM and blocks until completion.

Behind the scenes the SDK:
1. Serialises your DataFrame to JSONL and uploads it to a suitable compute node
2. Submits the finetune job and polls the compute node until the job finishes

The job specification pipeline needs values for these variables:

```
job = client.finetune(
    model="qwen2.5-0.5b",    # The model you choose to finetune
    training_mode="low",      # How much training: "low", "medium", or "high"
    data=data,                # Your dataset (DataFrame, HF Dataset, or file path)
    gpus=1,                   # Number of A100 GPUs to use
    time_limit="01:00:00",    # Maximum time your job should run for
    backend="cineca",         # HPC backend: "cineca" (default), "eversetech", "agh", or "acf"
)
```

In [ ]:
# Create and inspect the job (does not submit yet)
job = client.finetune(
    model="qwen2.5-0.5b",
    training_mode="low",     # QLoRA, 4-bit, rank 8
    data=dataset,
    gpus=1,
    time_limit="01:00:00",
)

cfg = job.spec.training_config
print(f"Job ID:       {job.job_id}")
print(f"Model:        {job.spec.model}")
print(f"Mode:         {job.spec.training_mode.value}")
print(f"GPUs:         {job.spec.gpus}")
print(f"LoRA rank:    {cfg.lora_r}")
print(f"Quantisation: {cfg.quantization_bits}-bit" if cfg.use_quantization else "Quantisation: off")
print(f"Batch size:   {cfg.batch_size}")
print(f"LR:           {cfg.learning_rate}")

Job ID:       46ef9be2
Model:        qwen2.5-0.5b
Mode:         low
GPUs:         1
LoRA rank:    8
Quantisation: 4-bit
Batch size:   2
LR:           0.0002


In [ ]:
# Submit to SLURM and wait for completion
result = job.run(wait=True, poll_interval=30)

print(f"\nResult: {result}")


Preparing finetune job: ft-qwen2.5-0.5b-46ef9be2
Uploading dataset (3 rows) to CINECA...
Uploading: /tmp/tmp7n1r01oa.jsonl -> $WORK/datasets/46ef9be2/train.jsonl
Upload complete: 0.00 MB in 4.4s
  Dataset: $WORK/datasets/46ef9be2/train.jsonl
  Container: $WORK/containers/afrilink-qwen2.5-0.5b.sif

Submitting to SLURM...
Job submitted: 34302214
  Name: ft-qwen2.5-0.5b-46ef9be2
  Model: qwen2.5-0.5b
  GPUs: 1
  Training mode: low

Waiting for job completion (timeout: 14400s)...

Job completed successfully!

Result: {'job_id': '46ef9be2', 'slurm_job_id': '34302214', 'status': 'completed', 'output_dir': '$WORK/finetune_outputs/46ef9be2', 'model_path': '$WORK/finetune_outputs/46ef9be2/', 'billing': {'total_gpu_minutes': 0.0, 'total_gpu_hours': 0.0, 'rate_per_gpu_hour': 2.0, 'total_cost_usd': 0.0}}


In [ ]:
# (Optional) Check logs while waiting or after completion
print(job.get_logs(tail=50))

  "learning_rate": 0.0002
}
Job Configuration:
{
  "model_path": "/workspace/models/qwen2.5-0.5b",
  "dataset_path": "/workspace/data/train.jsonl",
  "output_dir": "/workspace/output",
  "lora_r": 8,
  "lora_alpha": 16,
  "batch_size": 2,
  "gradient_accumulation_steps": 8,
  "num_epochs": 3,
  "learning_rate": 0.0002
}

Loading model from: /workspace/models/qwen2.5-0.5b
trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184

Loading dataset from: /workspace/data/train.jsonl

Starting training...
{'train_runtime': 3.1527, 'train_samples_per_second': 2.855, 'train_steps_per_second': 0.952, 'train_loss': 0.5028878847757975, 'epoch': 3.0}

Saving model to: /workspace/output

Training complete!

Listing output files:
total 29
drwxr-xr-x  5 iomunga0 interactive     4096 Feb 17 10:31 .
drwxr-xr-x 12 iomunga0 interactive     4096 Feb 17 10:31 ..
-rw-r--r--  1 iomunga0 interactive      683 Feb 17 10:31 adapter_config.json
-rw-r--r--  1 iomunga0 interactive  4350392 Feb 17

---
## 6. **Download Trained Model Weights after Finetune is done**

The adapter files (`adapter_model.safetensors`, `adapter_config.json`, tokenizer files) are downloaded flat into the target directory — ready for `PeftModel.from_pretrained()`.

Use `client.download_model(result["job_id"], MODEL_DIR)` with a pre-specified directory for where you'd like the model files to be saved in your notebook's file system.
The extra steps here are just to print out status logs.

In [ ]:
if result.get("status") != "completed":
    print(f"Job did not complete successfully (status: {result.get('status')})")
    print("Check logs with: job.get_logs()")
    if result.get("error"):
        print(f"\nError output:\n{result['error'][:500]}")
else:
    MODEL_DIR = f"./my-model-{result['job_id']}"

    client.download_model(result["job_id"], MODEL_DIR)

    # Verify downloaded files
    import os
    for f in sorted(os.listdir(MODEL_DIR)):
        size = os.path.getsize(os.path.join(MODEL_DIR, f))
        print(f"  {f:40} {size/1024:.1f} KB")

Download complete: 19.30 MB in 47.6s
Model ready at: ./my-model-46ef9be2
  Load with: PeftModel.from_pretrained(base_model, "./my-model-46ef9be2")
  README.md                                5.0 KB
  adapter_config.json                      0.7 KB
  adapter_model.safetensors                4248.4 KB
  added_tokens.json                        0.6 KB
  merges.txt                               1632.7 KB
  special_tokens_map.json                  0.6 KB
  tokenizer.json                           11154.5 KB
  tokenizer_config.json                    7.1 KB
  training_args.bin                        5.1 KB
  vocab.json                               2711.8 KB


---
## 7. **Load Adapter & Run Inference**

Load the base model, attach the LoRA adapter, and generate text. You can keep experimenting at this point, or finetune more models!

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# MODEL_DIR is set in cell 16 (only if job completed)
if 'MODEL_DIR' not in dir():
    raise RuntimeError("No model downloaded — the training job did not complete successfully.")

BASE_MODEL = "Qwen/Qwen2.5-0.5B"

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(base_model, MODEL_DIR)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print("Model loaded.")

Loading base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded.


In [ ]:
# Generate a response
prompt = (
    "Below is an instruction that describes a task.\n\n"
    "### Instruction:\nWhat is transfer learning?\n\n"
    "### Response:\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.7,
        do_sample=True,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Below is an instruction that describes a task.

### Instruction:
What is transfer learning?

### Response:
Transfer learning is a machine learning technique that involves training a model on one dataset and then applying the learned knowledge to a new, related dataset.

In transfer learning, the model architecture of the original model is used to extract features from the new dataset, and then the learned features are used to make predictions on the new dataset.

Transfer learning can be used to improve the performance of a model on a new dataset by reducing the amount of data needed for training. This is because the model architecture is already optimized for the original dataset, and the new dataset is similar to the original dataset in terms of features.

However, transfer learning may also cause overfit


---
## Utility: **Queue Status & Job Management**

Come back to this code cell to check the list of finetune jobs you've put on AfriLink's processing queue using `client.list_jobs()`

In [ ]:
# List your SLURM jobs
for j in client.list_jobs():
    print(j)

# Cancel a job (uncomment and set ID):
# client.cancel_job("SLURM_JOB_ID")

# Run arbitrary commands on CINECA:
code, out, err = client.run_command("squeue -u $USER")
print(out)

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)



---
## **Summary**

```
pip install afrilink-sdk

from afrilink import AfriLinkClient

client = AfriLinkClient()
client.authenticate()                                         # DataSpires + CINECA

job = client.finetune(model="qwen2.5-0.5b", data=df, gpus=1) # create job
result = job.run(wait=True)                                    # submit & wait

client.download_model(result["job_id"], "./my-model")         # download adapter

model = PeftModel.from_pretrained(base, "./my-model")         # inference
```

For the full docs, check out our live on platform on https://dataspires.com and sign up for an account to use AfriLink today!